# AtfxFile Usage Example

This notebook demonstrates how to use the `AtfxFile` class for simplified ATFX file access with JAQueL queries and pandas DataFrames.

## Setup

First, ensure you have wodson and its dependencies installed:

```bash
uv pip install wodson
# or if you're in the development environment:
uv sync
```

In [1]:
# Import required modules
from pathlib import Path

import pandas as pd

from wodson.atfx import AtfxFile

# Set pandas display options for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("✓ Imports successful")

✓ Imports successful


In [2]:
# Locate the example ATFX file from checked-in test data
# Using files from tests/data/openatfx which are version controlled
EXAMPLE_FILE = Path("../../tests/data/openatfx/asam600/Example_Simple.atfx")

if not EXAMPLE_FILE.exists():
    # Alternative location if running from a different directory
    EXAMPLE_FILE = Path("tests/data/openatfx/asam600/Example_Simple.atfx")

assert EXAMPLE_FILE.exists(), f"Example file not found: {EXAMPLE_FILE}"
print(f"Using example file: {EXAMPLE_FILE}")

Using example file: ..\..\tests\data\openatfx\asam600\Example_Simple.atfx


## 1. Opening a File and Exploring the Model

The `AtfxFile` class is a context manager that automatically handles file opening, model loading, and cleanup.

In [3]:
# Open the ATFX file and explore available entities
with AtfxFile(EXAMPLE_FILE) as atfx:
    # Query to see what entities are available
    # Note: We use query() rather than accessing model internals
    print("Querying for Environments:")
    envs = atfx.query({"AoEnvironment": {}, "$attributes": {"id": 1}})
    print(f"  Found {len(envs)} environment(s)")

    print("\nQuerying for Measurements:")
    meas = atfx.query({"AoMeasurement": {}, "$attributes": {"id": 1}})
    print(f"  Found {len(meas)} measurement(s)")

Querying for Environments:
  Found 1 environment(s)

Querying for Measurements:
  Found 1 measurement(s)


## 2. Simple JAQueL Queries

Use the `.query()` method to execute JAQueL queries and get pandas DataFrames. This is the primary way to access data from ATFX files.

In [6]:
# Query all environments
with AtfxFile(EXAMPLE_FILE) as atfx:
    environments = atfx.query({"AoEnvironment": {}})

    print("Environments:")
    display(environments)

Environments:


,Environment.Name,Environment.Id
0,MyEnvironment,90


In [7]:
# Query all measurements
with AtfxFile(EXAMPLE_FILE) as atfx:
    measurements = atfx.query({"AoMeasurement": {}, "$attributes": {"id": 1, "name": 1}})

    print(f"\nFound {len(measurements)} measurement(s):")
    display(measurements)


Found 1 measurement(s):


,Measurement.Id,Measurement.Name
0,93,MyMeasurement


## 3. Query Submatrix Entities

Submatrices contain the bulk measurement data organized into local columns.

In [8]:
# Query submatrices
with AtfxFile(EXAMPLE_FILE) as atfx:
    submatrices = atfx.query({"AoSubmatrix": {}, "$attributes": {"id": 1, "name": 1, "number_of_rows": 1}})

    print(f"\nFound {len(submatrices)} submatrix/submatrices:")
    display(submatrices)


Found 1 submatrix/submatrices:


,Submatrix.Id,Submatrix.Name,Submatrix.NumberOfRows
0,99,MyMeasurement,2


In [9]:
# Query local columns
with AtfxFile(EXAMPLE_FILE) as atfx:
    localcolumns = atfx.query({"AoLocalColumn": {}, "$attributes": {"id": 1, "name": 1}})

    print(f"\nFound {len(localcolumns)} local column(s):")
    display(localcolumns.head(10))


Found 5 local column(s):


,Localcolumn.Id,Localcolumn.Name
0,100,MyMqLong
1,101,MyMqString
2,102,MyMqFloat
3,103,MyMqDouble
4,104,MyMqTime


## 4. Bulk Time Series Data with `.timeseries()`

The `.timeseries()` method efficiently reads bulk signal data from Submatrix LocalColumns.

In [10]:
# Read specific columns using patterns
with AtfxFile(EXAMPLE_FILE) as atfx:
    submatrices = atfx.query({"AoSubmatrix": {}, "$attributes": {"id": 1}})

    if len(submatrices) > 0:
        id_col = [c for c in submatrices.columns if "Id" in c][0]
        submatrix_id = int(submatrices.iloc[0][id_col])

        # Read only columns matching patterns (supports wildcards)
        # For example, read all columns starting with specific prefixes
        try:
            filtered_df = atfx.timeseries(
                submatrix_id,
                column_patterns=["*"],  # All columns (you can use patterns like "Time*", "Speed*")
            )

            print("\nFiltered time series data (first 5 rows):")
            display(filtered_df.head())
        except Exception as e:
            print(f"Note: {e}")

Note: invalid literal for int() with base 10: 'explicit'


In [11]:
# First, find a submatrix ID to read from
with AtfxFile(EXAMPLE_FILE) as atfx:
    submatrices = atfx.query({"AoSubmatrix": {}, "$attributes": {"id": 1, "name": 1, "number_of_rows": 1}})

    if len(submatrices) > 0:
        # Extract the submatrix ID (column name includes entity prefix)
        id_col = [c for c in submatrices.columns if "Id" in c][0]
        submatrix_id = int(submatrices.iloc[0][id_col])

        print(f"Reading time series data from Submatrix ID: {submatrix_id}")
        print(f"Submatrix has {submatrices.iloc[0][[c for c in submatrices.columns if 'NumberOfRows' in c][0]]} rows\n")

        # Read all columns from this submatrix
        timeseries_df = atfx.timeseries(submatrix_id)

        print("Time series data:")
        display(timeseries_df.head(10))
        print(f"\nShape: {timeseries_df.shape}")
        print(f"Columns: {list(timeseries_df.columns)}")
    else:
        print("No submatrices found in this file")

Reading time series data from Submatrix ID: 99
Submatrix has 2 rows



ValueError: invalid literal for int() with base 10: 'explicit'

In [ ]:
# Try with AllTypes example from test data
ALLTYPES_FILE = Path("../../tests/data/openatfx/asam600/Example_AllTypes.atfx")

if not ALLTYPES_FILE.exists():
    ALLTYPES_FILE = Path("tests/data/openatfx/asam600/Example_AllTypes.atfx")

if ALLTYPES_FILE.exists():
    with AtfxFile(ALLTYPES_FILE) as atfx:
        # Query measurements
        measurements = atfx.query({"AoMeasurement": {}, "$attributes": {"id": 1, "name": 1}})

        print(f"Example_AllTypes.atfx contains {len(measurements)} measurement(s):")
        display(measurements)

        # Query submatrices with row counts
        submatrices = atfx.query({"AoSubmatrix": {}, "$attributes": {"name": 1, "number_of_rows": 1}})

        print("\nSubmatrices in Example_AllTypes.atfx:")
        display(submatrices)
else:
    print(f"Example_AllTypes.atfx not found at {ALLTYPES_FILE}")

## Summary

The `AtfxFile` class provides:

1. **Simple file opening** — Just pass a file path to the context manager
2. **DataFrame queries** — `.query()` method uses JAQueL syntax for easy metadata querying
3. **Bulk time series data** — `.timeseries()` method for efficient LocalColumn data access
4. **Low-level access** — `.con_i` property available for advanced operations if needed

### Key Methods

- **`query(query_dict)`** — Execute JAQueL queries, returns pandas DataFrame
  - Query any entity: Environment, Measurement, Submatrix, LocalColumn, etc.
  - Use filters, wildcards, and selection patterns
  
- **`timeseries(submatrix_id, column_patterns=None)`** — Read bulk signal data
  - Fast access to LocalColumn values within a Submatrix
  - Supports column pattern matching with wildcards
  - Returns data as pandas DataFrame with proper indexing

For more examples and detailed documentation, see:
- `docs/atfx-file.md` — Complete AtfxFile user guide
- `docs/USAGE.md` — Detailed guide to SelectStatement queries
- `docs/atfx.md` — Overview of all wodson.atfx APIs